***

Preparing Workspace

***

In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import os
import tqdm
import urllib.request, json


## Setting file paths ---

user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_code    = os.path.join(path_git, 'Data', 'Census')
    path_config0 = os.path.join(path_git , 'config')
    path_config  = os.path.join(path_code, 'config')


## User defined functions ---

exec(open(os.path.join(path_config0, 'Functions.py')).read())



***

Importing

***

In [ ]:

## CPS ---
# initialize empty list to store data frames
# iterate through each year
    # pull PUMS variables list from json file found on ACS website
    # convert to dictionary
    # convert to pandas data frame
    # apply year tag
    # append to list
# concatenate all data frames together

print(''); print('')
print('Importing CPS variables tables by year...'); print('')

year_start = 2009
year_end   = 2023
years = range(year_start, year_end+1)

list_df_cps = []

for year in tqdm(years):
    try:
        if year in [1995, 1997, 1999]:
            url_to_import = f"https://api.census.gov/data/{year}/cps/foodsec/apr/variables.json"
        if year in [1998]:
            url_to_import = f"https://api.census.gov/data/{year}/cps/foodsec/aug/variables.json"
        if year in [2000]:
            url_to_import = f"https://api.census.gov/data/{year}/cps/foodsec/sep/variables.json"
        if year in sequence(2001, 2023, 1):
            url_to_import = f"https://api.census.gov/data/{year}/cps/foodsec/dec/variables.json"
            
        with urllib.request.urlopen(url_to_import) as url:
    
            dict_cps = json.load(url)
            df_cps = pd.DataFrame.from_dict(dict_cps['variables']).T.reset_index().rename(columns = {'index':'ID'})
    
            df_cps['Year'] = year
            list_df_cps.append(df_cps)
    except Exception as e: print(e)
    
df_cps = pd.concat(list_df_cps)
df_cps = df_cps.reset_index(drop=True)
print(df_cps.shape)
display(df_cps.head())


# create empty list to store data frames
# iterate through each year
    # create empty list to store data frames
    # subset all variables to year
        # create empty list to store data frames
            # subset pums variables to one ID at a time
            # iterate through all key/value combinations in dictionaries that represent the value mappings to make pandas data frames 
            # store them in list of data frames
        # concatenate specific ID variable mappings together
        # add some labels, clean column names
    # apply year tag
    # convert to pandas data frame
    # store in list of data frames

# concatenate all data frames together


print(''); print('')
print('Cleaned CPS variables table:'); print('')

list_df_years = []

for year in tqdm(years):
    df_cps_vars = df_cps[df_cps['Year'] == year]
    
    list_df = []
    
    for ID in df_cps_vars['ID'].values:
        
        df_ID = df_cps_vars[df_cps_vars['ID'] == ID].reset_index(drop = True)

        if df_ID['values'][0] is np.nan:
            df_vars = pd.DataFrame()
            df_vars['Value1'] = np.nan
            df_vars['Value2'] = np.nan
            df_vars['Description'] = np.nan
            df_vars['ID'] = ID
            df_vars['Label'] = df_cps_vars[df_cps_vars['ID'] == ID].reset_index(drop=True)['label'].values[0]
            df_vars['Suggested Weight'] = df_cps_vars[df_cps_vars['ID'] == ID].reset_index(drop=True)['suggested-weight'].values[0]
            df_vars = df_vars[['Label', 'ID', 'Value1', 'Value2', 'Description', 'Suggested Weight']]

        else:
        
            for key in list(df_ID['values'][0].keys()):
                
                if key == 'item':
                    dict_values = {
                                     'Value1'     : list(list(df_ID['values'].values)[0]['item'].keys()  )
                                   , 'Value2'     : list(list(df_ID['values'].values)[0]['item'].keys()  )
                                   , 'Description': list(list(df_ID['values'].values)[0]['item'].values())
                                  }
                    df_vars = pd.DataFrame(dict_values)
                    
    
                if key == 'range':
                    
                    list_range = []
        
                    for value in df_ID['values'][0]['range']:
                        dict_values = {
                                        'Value1'     : [value['min']]
                                      , 'Value2'     : [value['max']]
                                      , 'Description': [value['description']]
                                     }
                        
                        list_range.append(pd.DataFrame(dict_values))
                        
                    df_vars = pd.concat(list_range)
                    
                df_vars['ID'] = ID
                df_vars['Label'] = df_cps_vars[df_cps_vars['ID'] == ID].reset_index(drop=True)['label'].values[0]
                df_vars['Suggested Weight'] = df_cps_vars[df_cps_vars['ID'] == ID].reset_index(drop=True)['suggested-weight'].values[0]
        
                df_vars = df_vars[['Label', 'ID', 'Value1', 'Value2', 'Description', 'Suggested Weight']]
        
        list_df.append(df_vars)
    
    
    df_cps_vars = pd.concat(list_df)
    df_cps_vars['Year'] = year
    
    list_df_years.append(df_cps_vars)

df_cps = pd.concat(list_df_years)
df_cps = df_cps.sort_values(['Year', 'ID', 'Value1'], ascending = [False, True, True])
df_cps = df_cps.reset_index(drop=True)
df_cps.head()

***

Exporting

***

In [ ]:


## Exporting to Git ---

workbook_name = 'Census Variable Tables.xlsx'

with pd.ExcelWriter(os.path.join(path_config, workbook_name),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
    df_cps.to_excel(writer, sheet_name = 'FOODSEC', index=False)

